In [1]:
import torch
import torch.nn as nn
import numpy as np
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## Prepare Data


In [2]:
bc = datasets.load_breast_cancer()
X, y = bc.data, bc.target

In [ ]:
n_samples, n_features = X.shape
print(f"Number of samples: {n_samples}, Number of features: {n_features}")

Number of samples: 569, Number of features: 30


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
X_train = torch.from_numpy(X_train.astype(np.float32))
X_test = torch.from_numpy(X_test.astype(np.float32))
y_train = torch.from_numpy(y_train.astype(np.float32))
y_test = torch.from_numpy(y_test.astype(np.float32))

In [8]:
y_train = y_train.view(y_train.shape[0], 1)
y_test = y_test.view(y_test.shape[0], 1)

## Setup Model


In [9]:
class LogisticRegression(nn.Module):
    def __init__(self, n_input_features: int):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(n_input_features, 1)

    def forward(self, x: torch.Tensor):
        y_predicted = torch.sigmoid(self.linear(x))
        return y_predicted

In [12]:
model = LogisticRegression(n_features)

## Setup Loss and Optimizer

In [13]:
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

## Training Loop

In [14]:
num_epochs = 100
for epoch in range(num_epochs):
    # Forward Pass
    y_predicted = model(X_train)
    loss = criterion(y_predicted, y_train)

    # Backward Pass
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()  # Reset gradients to zero

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [10/100], Loss: 0.6418
Epoch [20/100], Loss: 0.5102
Epoch [30/100], Loss: 0.4322
Epoch [40/100], Loss: 0.3813
Epoch [50/100], Loss: 0.3453
Epoch [60/100], Loss: 0.3181
Epoch [70/100], Loss: 0.2966
Epoch [80/100], Loss: 0.2792
Epoch [90/100], Loss: 0.2647
Epoch [100/100], Loss: 0.2523


## Evaluation

In [15]:
with torch.no_grad():
    y_predicted = model(X_test)
    y_predicted_cls = y_predicted.round()
    acc = y_predicted_cls.eq(y_test).sum() / float(y_test.shape[0])
    print(f"Accuracy: {acc:.4f}")

Accuracy: 0.9035
